In [7]:
import sys
from pathlib import Path
import numpy as np

root = Path.cwd()
candidates = [root, root.parent, root.parent.parent]
for base in candidates:
    if (base / "efgp_eigenpro_py").exists():
        sys.path.insert(0, str(base))
        break
else:
    raise RuntimeError("cannot find efgp_eigenpro_py from current path")

from efgp_eigenpro_py.eigenpro_precond import EigenProPreconditioner, build_preconditioner
from efgp_eigenpro_py.linear_solvers import pcg, richardson

np.set_printoptions(precision=3, suppress=True)


In [8]:
def expect_raises(fn, exc_type=ValueError):
    ok = False
    try:
        fn()
    except exc_type:
        ok = True
    if not ok:
        raise AssertionError("expected exception was not raised")


def assert_rel_close(got, ref, tol, label):
    err = np.linalg.norm(got - ref) / max(np.linalg.norm(ref), 1e-15)
    if err > tol:
        raise AssertionError(f"{label} relerr {err} > {tol}")
    return err


In [9]:
def test_precond_constructor_guards():
    U = np.eye(4, 2)
    theta = np.array([3.0, 2.0])

    expect_raises(lambda: EigenProPreconditioner(theta, U[:, 0], 1.0))
    expect_raises(lambda: EigenProPreconditioner(theta.reshape(2, 1), U, 1.0))
    expect_raises(lambda: EigenProPreconditioner(np.array([3.0]), U, 1.0))
    expect_raises(lambda: EigenProPreconditioner(np.array([3.0, -1.0]), U, 1.0))
    expect_raises(lambda: EigenProPreconditioner(theta, U, 0.0))
    expect_raises(lambda: EigenProPreconditioner(theta, U, 2.5))
    print("P1 ok")


def test_precond_matches_dense_formula():
    rng = np.random.default_rng(0)
    M, q = 7, 3
    Z = rng.normal(size=(M, q)) + 1j * rng.normal(size=(M, q))
    U, _ = np.linalg.qr(Z)
    theta = np.array([5.0, 3.0, 2.0])
    mu = 1.5

    P = build_preconditioner(theta, U, mu)
    scale = 1.0 - mu / theta
    P_dense = np.eye(M, dtype=complex) - U @ np.diag(scale) @ U.conj().T

    v = rng.normal(size=M) + 1j * rng.normal(size=M)
    got = P.apply(v)
    ref = P_dense @ v
    err = assert_rel_close(got, ref, 1e-12, "P2")
    print("P2 relerr =", err)


def test_precond_identity_on_orthogonal_complement():
    rng = np.random.default_rng(1)
    M, q = 8, 3
    Z = rng.normal(size=(M, q)) + 1j * rng.normal(size=(M, q))
    U, _ = np.linalg.qr(Z)
    theta = np.array([6.0, 4.0, 2.0])
    mu = 1.0
    P = build_preconditioner(theta, U, mu)

    v = rng.normal(size=M) + 1j * rng.normal(size=M)
    v = v - U @ (U.conj().T @ v)
    got = P.apply(v)
    err = np.linalg.norm(got - v) / max(np.linalg.norm(v), 1e-15)
    if err > 1e-12:
        raise AssertionError(f"P3 relerr {err} > 1e-12")
    print("P3 relerr =", err)


def test_precond_flattens_selected_eigendirections():
    rng = np.random.default_rng(2)
    M = 6
    Q, _ = np.linalg.qr(rng.normal(size=(M, M)))
    theta_all = np.array([9.0, 6.0, 3.0, 1.5, 1.0, 0.5])
    A = Q @ np.diag(theta_all) @ Q.T

    q = 3
    U = Q[:, :q]
    theta = theta_all[:q]
    mu = theta_all[q]
    P = build_preconditioner(theta, U, mu)

    for i in range(q):
        ui = U[:, i]
        got = P.apply(A @ ui)
        ref = mu * ui
        err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
        if err > 1e-12:
            raise AssertionError(f"P4 relerr {err} > 1e-12")
    print("P4 ok")


test_precond_constructor_guards()
test_precond_matches_dense_formula()
test_precond_identity_on_orthogonal_complement()
test_precond_flattens_selected_eigendirections()


P1 ok
P2 relerr = 8.657583053561291e-17
P3 relerr = 1.9750536820169732e-16
P4 ok


In [10]:
import scipy
print(scipy.__version__)

1.15.3


In [11]:
def make_spd(n, seed=0):
    rng = np.random.default_rng(seed)
    M = rng.normal(size=(n, n))
    A = M.T @ M + 0.5 * np.eye(n)
    return A


def test_pcg_matches_dense_solve():
    rng = np.random.default_rng(3)
    A = make_spd(12, seed=3)
    b = rng.normal(size=12)
    x_ref = np.linalg.solve(A, b)

    def matvec(v):
        return A @ v

    x, it, relres = pcg(matvec, b, tol=1e-10, maxiter=200, precond=None)
    relerr = np.linalg.norm(x - x_ref) / np.linalg.norm(x_ref)
    if relres >= 1e-10 or relerr >= 1e-8 or it <= 0:
        raise AssertionError("S1 failed")
    print("S1 relres =", relres, "relerr =", relerr, "it =", it)


def test_richardson_matches_dense_solve():
    rng = np.random.default_rng(4)
    A = make_spd(10, seed=4)
    b = rng.normal(size=10)
    x_ref = np.linalg.solve(A, b)

    def matvec(v):
        return A @ v

    eta = 0.9 / np.linalg.norm(A, 2)
    x0 = np.zeros_like(b)
    x, it, relres = richardson(matvec, b, x0, eta=eta, tol=1e-8, maxiter=5000)
    relerr = np.linalg.norm(x - x_ref) / np.linalg.norm(x_ref)
    if relres >= 1e-8 or relerr >= 1e-6 or it <= 0:
        raise AssertionError("S2 failed")
    print("S2 relres =", relres, "relerr =", relerr, "it =", it)


def test_exact_inverse_preconditioner():
    A = np.diag([1.0, 10.0, 100.0, 1000.0]).astype(float)
    b = np.array([1.0, -2.0, 3.0, -4.0])
    x_ref = np.linalg.solve(A, b)

    def matvec(v):
        return A @ v

    Ainv = np.diag(1.0 / np.diag(A))

    def precond(v):
        return Ainv @ v

    x_r, it_r, rr_r = richardson(matvec, b, np.zeros_like(b), eta=1.0, tol=1e-12, maxiter=10, precond=precond)
    err_r = np.linalg.norm(x_r - x_ref) / np.linalg.norm(x_ref)
    if it_r != 1 or rr_r >= 1e-12 or err_r >= 1e-12:
        raise AssertionError("S3 richardson failed")

    x_cg, it_cg, rr_cg = pcg(matvec, b, tol=1e-12, maxiter=10, precond=precond)
    err_cg = np.linalg.norm(x_cg - x_ref) / np.linalg.norm(x_ref)
    if it_cg > 2 or rr_cg >= 1e-12 or err_cg >= 1e-12:
        raise AssertionError("S3 pcg failed")
    print("S3 ok")


def test_richardson_zero_iter_and_exact_init():
    A = np.diag([2.0, 3.0, 5.0])
    b = np.array([2.0, 3.0, 5.0])

    def matvec(v):
        return A @ v

    x_ref = np.linalg.solve(A, b)
    x, it, rr = richardson(matvec, b, x_ref, eta=0.1, tol=1e-12, maxiter=0)
    if it != 0 or rr >= 1e-12 or np.linalg.norm(x - x_ref) >= 1e-12:
        raise AssertionError("S4 failed")
    print("S4 ok")


def test_complex_hermitian_case():
    rng = np.random.default_rng(5)
    Z = rng.normal(size=(6, 6)) + 1j * rng.normal(size=(6, 6))
    A = Z.conj().T @ Z + 0.5 * np.eye(6)
    b = rng.normal(size=6) + 1j * rng.normal(size=6)
    x_ref = np.linalg.solve(A, b)

    def matvec(v):
        return A @ v

    x_cg, it_cg, rr_cg = pcg(matvec, b, tol=1e-10, maxiter=300)
    err_cg = np.linalg.norm(x_cg - x_ref) / np.linalg.norm(x_ref)
    if rr_cg >= 1e-10 or err_cg >= 1e-8:
        raise AssertionError("S5 pcg failed")

    eta = 0.9 / np.linalg.norm(A, 2)
    x_r, it_r, rr_r = richardson(matvec, b, np.zeros_like(b), eta, 1e-8, 5000)
    err_r = np.linalg.norm(x_r - x_ref) / np.linalg.norm(x_ref)
    if rr_r >= 1e-8 or err_r >= 1e-6:
        raise AssertionError("S5 richardson failed")
    print("S5 ok")


def test_preconditioning_improves_richardson():
    n = 20
    lam = np.logspace(0, 6, n)
    A = np.diag(lam)
    b = np.ones(n)

    def matvec(v):
        return A @ v

    q = 5
    idx = np.arange(n - q, n)
    U = np.eye(n)[:, idx]
    theta = lam[idx]
    mu = lam[n - q - 1]
    P = build_preconditioner(theta, U, mu).apply

    eta_plain = 0.9 / lam.max()
    _, _, rr_plain = richardson(matvec, b, np.zeros_like(b), eta_plain, 1e-12, 50, precond=None)

    eta_prec = 0.9 / mu
    _, _, rr_prec = richardson(matvec, b, np.zeros_like(b), eta_prec, 1e-12, 50, precond=P)

    if not (rr_prec < rr_plain):
        raise AssertionError("S6 failed")
    print("S6 rr_plain =", rr_plain, "rr_prec =", rr_prec)


def _version_tuple(v):
    parts = v.split(".")
    out = []
    for p in parts:
        num = ""
        for ch in p:
            if ch.isdigit():
                num += ch
            else:
                break
        if num == "":
            num = "0"
        out.append(int(num))
    return tuple(out)


def _have_scipy_cg():
    import importlib.metadata as imd
    np_ver = _version_tuple(np.__version__)
    try:
        scipy_ver_str = imd.version("scipy")
    except imd.PackageNotFoundError:
        print("scipy not installed: skip pcg-related tests")
        return False
    scipy_ver = _version_tuple(scipy_ver_str)
    if np_ver >= (2, 3, 0) and scipy_ver < (1, 15, 0):
        print("scipy", scipy_ver_str, "is too old for numpy", np.__version__)
        print("upgrade scipy to >=1.15 or downgrade numpy<2.3")
        return False
    try:
        from scipy.sparse.linalg import cg  # noqa: F401
        return True
    except BaseException as exc:
        print("scipy not usable, skip pcg-related tests:", exc)
        return False

if _have_scipy_cg():
    test_pcg_matches_dense_solve()
    test_exact_inverse_preconditioner()
    test_complex_hermitian_case()
else:
    print("skip pcg-related tests")

test_richardson_matches_dense_solve()
test_richardson_zero_iter_and_exact_init()
test_preconditioning_improves_richardson()


S1 relres = 1.5943035312308117e-13 relerr = 1.0761391005133558e-13 it = 14
S3 ok
S5 ok
S2 relres = 9.855405113055886e-09 relerr = 9.37003947703868e-08 it = 1077
S4 ok
S6 rr_plain = 0.7909159006326651 rr_prec = 0.6129464082179775
